# 03 — Figures & tables (regenerate *everything* from saved results)

Rebuilds **every** figure and table from the CSVs in `results/`. It does **no**
OSM fetching or metric computation, so it is safe to run any time and fast to
iterate on plot styling.

> ⚠️ **Restart the kernel first** (Kernel → Restart & Run All) whenever you've
> edited code under `src/cycleform/`. Jupyter caches imported modules, so without
> a restart you silently run the *old* plotting code — that is what causes
> "missing plots", "only one labelled", or leftover PDFs.

Everything is **PNG** (300 dpi) in `results/figures/`. Per-city plots are written
in two variants: `<name>.png` (points coloured by nation) and
`<name>_labeled.png` (the fixed exemplar cities — Newcastle, Münster, Oslo, … —
labelled).

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
from IPython.display import Image, display
from cycleform import report, figures, scenarios
from cycleform.config import settings
pd.set_option('display.width', 200, 'display.max_columns', 40)
FIGDIR = settings.results / 'figures'
print('figures ->', FIGDIR)

## 1. Load the saved tables

In [ ]:
wide  = report.load_wide()
table = report.load_analysis()
print(wide.shape, '| places with a cycling rate:', table['value'].notna().sum(),
      '| UK:', int(wide['is_uk'].sum()))
wide[['place_id','country','is_uk','bikeable_length_share','components_per_km_bike']].head()

## 2. Summary tables

`components_per_km_bike` = disconnected pieces per km of cycle network (size-
normalised fragmentation of the current cycle network).

In [ ]:
tabs = report.summary_tables()
tabs['correlations'].head(25)   # every metric vs cycling rate (Spearman), ranked by |rho|

## 3. Regenerate ALL figures

One call draws: the typology (±labels), the all-metric correlation bar, the
top-correlates grid, the metric–metric heatmap, an **individual scatter for every
metric vs cycling rate** (±labels), the ranked dot plots, and the bike-vs-road
panels (±labels). `refresh_first=True` first rebuilds the tables from the per-place
files, so newly-finished places are included.

In [ ]:
paths = report.make_figures(refresh_first=True)
labeled = sum(str(p).endswith('_labeled.png') for p in paths)
scatters = sum('outcome_vs_' in p.name for p in paths)
print(f'wrote {len(paths)} figures  ({scatters} metric-vs-cycling scatters, {labeled} labelled variants)')
print('PDFs in figures dir:', len(list(FIGDIR.glob('*.pdf'))), '(should be 0)')

## 4. Predictive model figures + tables

In [ ]:
res = report.make_model_report()   # pred-vs-actual (±labels) + feature importance
display(res['performance'])
res['importance'].head(15)

## 5. Grown-network scenario (optional)

Only runs if you've already produced scenario results with `python run_scenarios.py`
(see `SCENARIOS.md`). Draws the metric-shift, predicted-rate-shift and form-space
movement figures for the Tyne & Wear boroughs.

In [ ]:
if not scenarios.build_scenario_table().empty:
    sres = report.make_scenario_report()
    print('scenario figures:', [p.name for p in sres['figures']])
    display(sres['predictions'])
else:
    print('No scenario results yet — run `python run_scenarios.py` first (see SCENARIOS.md).')

## 6. Preview a few (everything is on disk in `results/figures/`)

In [ ]:
Image(str(FIGDIR / 'correlations_vs_cycling.png'))

In [ ]:
Image(str(FIGDIR / 'outcome_vs_bikeable_length_share_labeled.png'))

In [ ]:
Image(str(FIGDIR / 'outcome_vs_components_per_km_bike_labeled.png'))

In [ ]:
# full inventory
figs = sorted(p.name for p in FIGDIR.glob('*.png'))
print(len(figs), 'PNGs |', sum(n.endswith('_labeled.png') for n in figs), 'labelled |',
      len(list(FIGDIR.glob('*.pdf'))), 'PDFs')
figs[:40]

## 7. Plain-text results digest 

Writes `results/report.md` — a self-describing markdown summary of the dataset, correlations, model, contrasts, typology and caveats. Reads saved tables only (fast); regenerate any time. 

In [ ]:
p = report.text_report()
print('wrote', p)
print(p.read_text(encoding='utf-8')[:2000])